# Batch Plan Designer

Writes a **batch plan**: a JSON recipe for a simulation batch. It lists how to *build* a zoo
(factory names, kwargs, counts, one root seed) and the simulation knobs. It never holds a graph.

Two things this buys over `design_zoo.ipynb`:

- **Memory.** This kernel builds nothing, so designing a 50k-graph batch costs a few KB instead of
  gigabytes. `design_zoo.ipynb` held the zoo twice: once as it was built, once again when
  `submit_jobs` reloaded it to write the shards.
- **Replay.** The plan stored beside the results is the recipe that produced them. The old
  `zoo_config` record could not replay: no factory writes `directed` into `PopulationGraph.params`,
  and two of them use key names that differ from their own kwargs.

Submission happens in a terminal, not here. The last section prints the command.

## Section 0 - Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

from moran_process.pipeline.batch_plan import (
    spec, make_plan, save_plan, describe, project_root, verify_reproducible,
)

PROJECT_ROOT = project_root()
print(f"Project root: {PROJECT_ROOT}")

## Section 1 - Batch identity and simulation knobs

`ZOO_SEED` controls which random *topologies* exist. `BATCH_SEED` controls the simulation RNG.
They are independent: re-running the same zoo with a new `BATCH_SEED` is a fresh Monte Carlo
sample of the same population structures.

In [ ]:
BATCH_NAME  = "2026_08_26-batch-plan-toy"   # <-- change for each experiment
DESCRIPTION = "Let's see if this batch plan works"
NOTES       = ""

R_VALUES   = [1, 1.1, 1.5, 2]   # selection coefficients
N_REPEATS  = 100         # simulations per graph per r
N_JOBS     = 10
ZOO_SEED   = 43                 # graph topology RNG
BATCH_SEED = 43                 # simulation RNG
ENGINE     = "cpp"              # "cpp" or "python"
QUEUE      = "gsla-cpu"
MEMORY     = "1GB"

## Section 2 - Named graphs

One `spec(...)` per graph. The kwargs go to the `PopulationGraph` factory verbatim, so
`directed=True` is recorded here even though the factory does not put it in `params`.

A typo in a factory name raises **now**, in this cell, rather than inside an LSF job later.

In [ ]:
SPECS = [
    spec("mammalian_lung_graph", branching_factor=2, depth=4, directed=True),
    spec("avian_graph",          n_rods=10, rod_length=3, directed=True),
    spec("fish_graph",           n_rods=3, rod_length=3, fillaments=4),
    spec("cycle_graph",          n_nodes=31, directed=True),
    spec("line_graph",           n_nodes=31, directed=True),
    spec("star_graph",           n_nodes=10, directed=True),
    spec("grid_graph",           width=8, height=4),
    spec("complete_graph",       n_nodes=10),
]
len(SPECS)

## Section 3 - Random null model

The sweep loop lives here, in Python; the plan stays a flat list of specs. Each spec gets its own
RNG stream, spawned from `ZOO_SEED` by spec index, so **appending** a spec leaves every earlier
spec's graphs bit-identical. Inserting one in the middle shifts everything after it.

In [ ]:
N_NODES     = [30]
EDGE_OFFSET = [-1, 0, 1, 2, 3, 4]   # n_edges = n_nodes + offset
PER_CONFIG  = 20

for n_nodes in N_NODES:
    for offset in EDGE_OFFSET:
        SPECS.append(
            spec("random_connected_graph",
                 count=PER_CONFIG,
                 n_nodes=n_nodes,
                 n_edges=n_nodes + offset)
        )

len(SPECS)

## Section 4 - Review and save

`describe` costs the batch without building anything. Check `simulations/job` before you submit:
that number, not the graph count, is what sets wall clock.

In [ ]:
plan = make_plan(
    batch_name=BATCH_NAME,
    specs=SPECS,
    r_values=R_VALUES,
    n_repeats=N_REPEATS,
    n_jobs=N_JOBS,
    zoo_seed=ZOO_SEED,
    batch_seed=BATCH_SEED,
    engine=ENGINE,
    queue=QUEUE,
    memory=MEMORY,
    description=DESCRIPTION,
    notes=NOTES,
)

describe(plan)

In [ ]:
plan_path = PROJECT_ROOT / "batch_plans" / f"{BATCH_NAME}.json"
save_plan(plan, plan_path)

## Section 5 - Optional: eyeball the zoo

One representative per spec. Because each spec draws from its own RNG stream spawned off
`ZOO_SEED`, the graph drawn here is literally the first graph that spec contributes to the real
batch, not a lookalike.

The size guard runs on the *built* graph's `n_nodes`, not on a kwarg: `mammalian_lung_graph`,
`avian_graph`, `fish_graph` and `grid_graph` do not take `n_nodes` at all, so there is nothing to
read off the spec. Building one of each is cheap even at batch scale; drawing a 1000-node graph
is not, and is unreadable anyway.

In [ ]:
import matplotlib.pyplot as plt
from moran_process.core.graph_zoo import GraphZoo
from moran_process.pipeline.batch_plan import build_zoo, make_plan

MAX_PREVIEW_NODES = 200

preview_specs = [{**s, "count": 1} for s in SPECS]
preview = make_plan("preview", preview_specs, R_VALUES, N_REPEATS, N_JOBS, zoo_seed=ZOO_SEED)

zoo = GraphZoo(name=f"{BATCH_NAME} (one per spec)")
skipped = []
for g in build_zoo(preview, progress_every=0):
    if g.n_nodes <= MAX_PREVIEW_NODES:
        zoo.add(g)
    else:
        skipped.append(g)

print(f"drawing {len(zoo)} of {len(SPECS)} specs")
if skipped:
    print(f"skipped (> {MAX_PREVIEW_NODES} nodes): "
          + ", ".join(f"{g.name} n={g.n_nodes}" for g in skipped))

zoo.draw_all(cols=3)

In [ ]:
# delete me 
len(zoo.graphs)

In [ ]:
# Determinism check: builds the zoo TWICE and compares wl_hashes.
# Run it on a new plan *shape*, not on every submission, and only when the plan is small.
verify_reproducible(plan)

## Section 6 - Submit

**Do not submit from this kernel.** Building the zoo is the memory-heavy step, and the VS Code
Run button executes on the WEXAC *login node*, where the watchdog kills CPU-heavy processes
(exit 144). Open an `inode` session and run the command the next cell prints.

`--dry-run` prints the plan and its cost, builds nothing, submits nothing.

In [ ]:
print(f"""
# In an inode session, from {PROJECT_ROOT}:

uv run python -m moran_process.pipeline.batch_plan --plan {plan_path} --dry-run
uv run python -m moran_process.pipeline.batch_plan --plan {plan_path}
""")